# 01 — Comptage de lignes (NCLOC, SLOC)

**Auteur :** Benoît Moraillon — **Licence :** AGPL-3.0

Le comptage de lignes est la métrique la plus simple et la plus universelle. Elle est aussi la plus mal comprise — il existe au moins quatre façons de compter les lignes de code, et leur différence compte.

## Définitions retenues

| Métrique | Définition |
|---|---|
| `total` | Lignes physiques (vides, commentaires et code confondus) |
| `blank` | Lignes ne contenant que des espaces / tabulations |
| `comment` | Lignes ne contenant qu'un commentaire (pas de code mêlé) |
| `sloc` | Source LOC = `total - blank` |
| `ncloc` | Non-Comment LOC = `sloc - comment` |

Le **NCLOC** est l'indicateur métier le plus utile : il reflète la quantité de logique réelle. Une ligne `x = 1  # comment` est comptée comme NCLOC ET SLOC : elle porte du code, le commentaire est un bonus.

Conventions tree-sitter conformes à cloc / SonarQube.

In [ ]:
# Bootstrap : permet l'import des modules sans installation pip
import sys
from pathlib import Path

root = Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import setup_paths
setup_paths.setup()

## Exemple progressif

Commençons par un fichier minimal.

In [ ]:
from kodoneko_metrics import count_lines

source_simple = '''
def hello():
    return "world"
'''

r = count_lines(source_simple, language="python")
print(r)

On a 3 lignes physiques utiles + une ligne vide en tête (la première du triple-string).

## Ajoutons des commentaires et docstrings

**Convention KodoNeko (depuis v1.1)** : les docstrings sont comptées comme commentaires, pas comme code. Pour le comportement cloc historique, passer `exclude_docstrings=False`.

In [ ]:
source_documente = """
def average(values):
    \"\"\"Retourne la moyenne arithmétique.\"\"\"
    # cas particulier : liste vide
    if not values:
        return 0.0
    return sum(values) / len(values)
"""

# Convention KodoNeko (défaut)
r = count_lines(source_documente, language="python")
print(f"NCLOC (docstrings exclus) : {r.ncloc}")
print(f"Comment                    : {r.comment}")

# Convention cloc
r_cloc = count_lines(source_documente, language="python", exclude_docstrings=False)
print(f"NCLOC (cloc-like)          : {r_cloc.ncloc}")

La docstring fait passer le NCLOC de 6 (cloc) à 5 (KodoNeko) : on considère que documenter n'est pas coder.

## Ce que NCLOC ne dit PAS

Le NCLOC est plat : il ne distingue pas un fichier de 200 lignes bien structuré d'un fichier de 200 lignes mal écrit. C'est pour ça qu'on a besoin des trois métriques suivantes (Halstead, McCabe, Understandability).

In [ ]:
# Comparaison : deux fichiers de NCLOC identique mais qualité différente
bon = '''
def somme(a, b):
    return a + b

def produit(a, b):
    return a * b

def difference(a, b):
    return a - b
'''

mauvais = '''
def f(x, y, z, w):
    if x: 
        if y: 
            if z: 
                if w: return 1
                else: return 2
            else: return 3
        else: return 4
    return 5
'''

print(f"Bon code     : NCLOC = {count_lines(bon, language='python').ncloc}")
print(f"Mauvais code : NCLOC = {count_lines(mauvais, language='python').ncloc}")

Les deux fichiers ont un NCLOC similaire, pourtant l'un est trivial et l'autre est un cauchemar à comprendre. C'est l'**understandability** (notebook 04) qui fera la différence.

## API de référence

| Fonction | Retour |
|---|---|
| `count_lines(source, language=...)` | `LineCounts` (total, blank, comment, sloc, ncloc) |

Cf. notebook **02** pour Halstead, **03** pour McCabe, **04** pour Understandability.

---

# Mesurer sur un dépôt réel, et dans le temps

Jusqu'ici, nous avons compté les lignes d'un fichier isolé. Mais un projet
réel, c'est des centaines de fichiers qui grossissent, maigrissent et se
réorganisent au fil des mois. Cette dernière partie déroule le comptage de
lignes sur un **vrai dépôt open-source**, puis observe comment sa taille a
évolué dans le temps — car une métrique ne prend tout son sens qu'en mouvement.

> ⚠️ Cette section clone un dépôt public (nécessite un accès réseau) et, pour
> les langages autres que Python, requiert `tree-sitter`. Sans ces prérequis,
> les cellules affichent un message explicatif et s'interrompent proprement,
> sans erreur.

## 1. Récupérer un dépôt public

Nous clonons un petit projet open-source réel. Le clone est *idempotent* : si le
dépôt est déjà présent localement, on le réutilise sans le retélécharger.

In [ ]:
import subprocess, tempfile
from pathlib import Path

REPO_URL = "https://github.com/gothinkster/django-realworld-example-app.git"
repo_dir = Path(tempfile.gettempdir()) / "kodoneko_demo_repo"

if repo_dir.exists():
    print("Dépôt déjà présent :", repo_dir)
else:
    print("Clonage en cours...")
    res = subprocess.run(
        ["git", "clone", REPO_URL, str(repo_dir)],
        capture_output=True, text=True,
    )
    if res.returncode == 0:
        print("Cloné :", repo_dir)
    else:
        print("Clone impossible (réseau indisponible ?) :")
        print("  ", res.stderr.strip().splitlines()[-1] if res.stderr.strip() else "erreur inconnue")
        repo_dir = None

## 2. Analyser le dépôt complet (instantané)

Premier réflexe : prendre une photographie de l'état actuel. On mesure le nombre de lignes de code (NCLOC)
sur l'ensemble du dépôt, tel qu'il est à l'instant présent (le dernier commit).

In [ ]:
from kodoneko_scanner import scan_repo

if repo_dir:
    report = scan_repo(repo_dir)
    print(f"Fichiers analysés : {report.files_scanned}")
    print(f"  NCLOC total       : {report.totals.ncloc}")
    print(f"  Lignes de commentaire : {report.totals.comment}")
    print(f"  Lignes vides          : {report.totals.blank}")
else:
    print("(dépôt indisponible — étape ignorée)")

## 3. Analyser un commit précis

Le moteur temporel `kodoneko_temporal` sait reconstituer l'état du dépôt à
**n'importe quelle référence git** (un tag, une branche, un SHA, ou une
expression comme `HEAD~5`). Ici, on mesure NCLOC à ce point de l'histoire, cinq commits
en arrière — un véritable voyage dans le passé du code.

In [ ]:
from kodoneko_temporal import analyze_at_ref

def mesure(path):
    """Applique notre métrique à un état du dépôt."""
    report = scan_repo(path)
    return report.totals.ncloc

if repo_dir:
    try:
        point = analyze_at_ref(repo_dir, "HEAD~5", analyzer=mesure)
        print(f"À {point.label} (commit {point.sha[:8]}) : NCLOC = {point.result}")
    except Exception as e:
        print(f"Commit indisponible (historique trop court ?) : {e}")
else:
    print("(dépôt indisponible — étape ignorée)")

## 4. Mesurer un écart entre deux moments (*diff temporel*)

La question la plus parlante n'est pas « combien ? » mais « **combien de plus
qu'avant ?** ». En comparant la mesure à deux références, on quantifie ce qu'un
intervalle de développement a réellement produit comme NCLOC.

In [ ]:
if repo_dir:
    try:
        avant = analyze_at_ref(repo_dir, "HEAD~10", analyzer=mesure)
        apres = analyze_at_ref(repo_dir, "HEAD", analyzer=mesure)
        delta = apres.result - avant.result
        signe = "+" if delta >= 0 else ""
        # Variation relative en pourcentage (2 décimales), protégée contre la division par zéro
        if avant.result:
            pct = delta / avant.result * 100
            pct_txt = f"{signe}{pct:.2f} %"
        else:
            pct_txt = "n/a (valeur initiale nulle)"
        print(f"NCLOC il y a 10 commits : {avant.result}")
        print(f"NCLOC aujourd'hui       : {apres.result}")
        print(f"Variation absolue    : {signe}{delta}")
        print(f"Variation relative   : {pct_txt}")
    except Exception as e:
        print(f"Diff indisponible : {e}")
else:
    print("(dépôt indisponible — étape ignorée)")

### Voir la taille respirer dans le temps

Un instantané ne dit pas si un projet grossit sainement ou enfle. En rejouant
la mesure à la fin de chaque mois, on transforme un chiffre figé en une
**trajectoire** : on voit le code naître, croître, parfois se contracter lors
d'un grand nettoyage.

In [ ]:
from kodoneko_temporal import analyze_over_windows

if repo_dir:
    serie = analyze_over_windows(repo_dir, analyzer=mesure, strategy="monthly")
    print(f"NCLOC à la fin de chaque mois ({len(serie)} fenêtres) :\n")
    maxv = max((p.result for p in serie), default=1) or 1
    precedent = None
    for point in serie:
        barre = "█" * min(40, int(point.result / maxv * 40))
        # Variation par rapport au mois précédent, en % (2 décimales)
        if precedent is None:
            var_txt = "     —  "
        elif precedent:
            pct = (point.result - precedent) / precedent * 100
            var_txt = f"{pct:+6.2f} %"
        else:
            var_txt = "    n/a "
        print(f"  {point.label}  {point.result:>8}  {var_txt}  {barre}")
        precedent = point.result
else:
    print("(dépôt indisponible — étape ignorée)")

## En résumé

Nous venons de parcourir les quatre façons d'interroger le nombre de lignes de code (NCLOC) :

1. **L'instantané** — l'état présent du dépôt entier.
2. **Le commit précis** — la mesure à un point quelconque de l'histoire.
3. **Le diff temporel** — l'écart entre deux moments, qui isole ce qu'une
   période a produit.
4. **La série temporelle** — la trajectoire complète, qui révèle les tendances.

Le même analyseur (`mesure`) a servi pour les quatre : c'est toute la force du
moteur temporel, qui découple *ce qu'on mesure* de *quand on le mesure*.